# Problem 2: Mars Rover Sample Collection Planning using PDDL

**Group members**

| Name | Roll No. |
|------|----------|
| Bodike Chaithali | B24CS013 |
| Kotapati Sai Mounika | B24CS019 |
| Bailapudi Kusuma Teja | B24CS011 |
| Lanka Devi Satwika | B24DS013 |
| Borada Jayasurya | B24MT010 |

**Approach:** `Domain.pddl`, `Problem.pddl` and `Problem_extended.pddl` are written to disk, parsed, grounded to STRIPS, and solved with
(1) Greedy Best-First Search with the h_add heuristic (satisficing),
(2) A* with the admissible LM-cut heuristic (optimal), and
(3) Breadth-First Search (optimal cross-check, scenario 1 only).
Plans are then validated by step-by-step simulation.

In [1]:
from pathlib import Path

DOMAIN = """(define (domain mars-rover)
  (:requirements :strips :typing)
  (:types location rover bay rock drill)

  (:predicates
    (rover-at ?r - rover ?l - location)
    (rock-at ?k - rock ?l - location)
    (drill-at ?d - drill ?l - location)
    (connected ?from - location ?to - location)
    (soft ?k - rock)
    (hard ?k - rock)
    (has-drill ?r - rover ?d - drill)
    (bay-empty ?b - bay)
    (bay-holds ?b - bay ?k - rock)
  )

  (:action move
    :parameters (?r - rover ?from - location ?to - location)
    :precondition (and (rover-at ?r ?from) (connected ?from ?to))
    :effect (and (not (rover-at ?r ?from)) (rover-at ?r ?to))
  )

  (:action pick-drill
    :parameters (?r - rover ?d - drill ?l - location)
    :precondition (and (rover-at ?r ?l) (drill-at ?d ?l))
    :effect (and (not (drill-at ?d ?l)) (has-drill ?r ?d))
  )

  (:action collect-soft
    :parameters (?r - rover ?k - rock ?l - location ?b - bay)
    :precondition (and (rover-at ?r ?l) (rock-at ?k ?l) (soft ?k) (bay-empty ?b))
    :effect (and (not (rock-at ?k ?l)) (not (bay-empty ?b)) (bay-holds ?b ?k))
  )

  (:action collect-hard
    :parameters (?r - rover ?k - rock ?l - location ?b - bay ?d - drill)
    :precondition (and (rover-at ?r ?l) (rock-at ?k ?l) (hard ?k) (bay-empty ?b) (has-drill ?r ?d))
    :effect (and (not (rock-at ?k ?l)) (not (bay-empty ?b)) (bay-holds ?b ?k))
  )

  (:action deposit
    :parameters (?r - rover ?k - rock ?l - location ?b - bay)
    :precondition (and (rover-at ?r ?l) (bay-holds ?b ?k))
    :effect (and (not (bay-holds ?b ?k)) (bay-empty ?b) (rock-at ?k ?l))
  )
)
"""

PROBLEM = """(define (problem mars-rover-1)
  (:domain mars-rover)
  (:objects
    base sitea siteb sitec sited sitee sitef - location
    curiosity - rover
    bay1 bay2 - bay
    rock1 rock2 rock3 rock4 rock5 - rock
    drill1 - drill)

  (:init
    (rover-at curiosity base)
    (drill-at drill1 sitee)
    (rock-at rock1 sitea) (rock-at rock2 siteb) (rock-at rock3 sitec)
    (rock-at rock4 sited) (rock-at rock5 sitef)
    (soft rock1) (soft rock2) (hard rock3) (soft rock4) (hard rock5)
    (bay-empty bay1) (bay-empty bay2)
    (connected base sitea) (connected sitea base)
    (connected base siteb) (connected siteb base)
    (connected sitea sitec) (connected sitec sitea)
    (connected siteb sited) (connected sited siteb)
    (connected sitec sitee) (connected sitee sitec)
    (connected sited sitee) (connected sitee sited)
    (connected sitee sitef) (connected sitef sitee)
    (connected sitec sited))

  (:goal (and
    (rock-at rock1 base) (rock-at rock2 base) (rock-at rock3 base)
    (rock-at rock4 base) (rock-at rock5 base)
    (bay-empty bay1) (bay-empty bay2)
    (rover-at curiosity base)))
)
"""

EXT = """(define (problem mars-rover-extended)
  (:domain mars-rover)
  (:objects
    base sitea siteb sitec sited sitee sitef - location
    curiosity - rover
    bay1 bay2 - bay
    rock1 rock2 rock3 rock4 rock5 rock6 rock7 - rock
    drill1 - drill)

  (:init
    (rover-at curiosity base)
    (drill-at drill1 sitee)
    (rock-at rock1 sitea) (rock-at rock2 siteb) (rock-at rock3 sitec)
    (rock-at rock4 sited) (rock-at rock5 sitef)
    (rock-at rock6 sitee) (rock-at rock7 sitef)
    (soft rock1) (soft rock2) (hard rock3) (soft rock4) (hard rock5)
    (soft rock6) (hard rock7)
    (bay-empty bay1) (bay-empty bay2)
    (connected base sitea) (connected sitea base)
    (connected sitea sitec) (connected sitec sitea)
    (connected siteb sited) (connected sited siteb)
    (connected sitec sitee) (connected sitee sitec)
    (connected sited sitee) (connected sitee sited)
    (connected sitee sitef) (connected sitef sitee)
    (connected sitec sited))

  (:goal (and
    (rock-at rock1 base) (rock-at rock2 base) (rock-at rock3 base)
    (rock-at rock4 base) (rock-at rock5 base) (rock-at rock6 base)
    (rock-at rock7 base)
    (bay-empty bay1) (bay-empty bay2)
    (rover-at curiosity base)))
)
"""

Path("Domain.pddl").write_text(DOMAIN)
Path("Problem.pddl").write_text(PROBLEM)
Path("Problem_extended.pddl").write_text(EXT)
print("Written:", [p for p in ["Domain.pddl", "Problem.pddl", "Problem_extended.pddl"] if Path(p).exists()])

Written: ['Domain.pddl', 'Problem.pddl', 'Problem_extended.pddl']


## Modelling notes (Tasks 1 and 2)

* **Requirements:** only `:strips` and `:typing`. There are no negative preconditions and no numeric fluents.
* **Types:** `location, rover, bay, rock, drill`.
* **Hard and soft rocks:** these are the static facts `(soft ?k)` and `(hard ?k)`. The action `collect-hard` additionally requires `(has-drill ?r ?d)`, while `collect-soft` does not.
* **Drill:** `pick-drill` deletes `(drill-at ...)` and adds `(has-drill ...)`. No action ever drops the drill, so the rover keeps it for the rest of the mission.
* **Bays:** `bay-empty` and `bay-holds` ensure each bay holds one rock, so at most two rocks can be carried.
* **Connectivity:** `connected` is directional. Two-way routes are stated in both directions, and the one-way slope is only `(connected sitec sited)`.
* **Extended scenario:** the dust storm removes both `connected` facts of Base and SiteB. Rock6 and Rock7 are added, and the goal covers Rock1 to Rock7.

In [2]:
import re, time, heapq, itertools
from collections import deque

def tokenize(text):
    text = re.sub(r";.*", "", text.lower())
    return re.findall(r"\(|\)|[^\s()]+", text)

def read_sexp(tokens):
    stack, cur = [], []
    for t in tokens:
        if t == "(":
            stack.append(cur); cur = []
        elif t == ")":
            done = cur; cur = stack.pop(); cur.append(done)
        else:
            cur.append(t)
    return cur[0]

def typed_list(items):
    out, pending, i = [], [], 0
    while i < len(items):
        if items[i] == "-":
            typ = items[i + 1]
            out += [(n, typ) for n in pending]; pending = []; i += 2
        else:
            pending.append(items[i]); i += 1
    out += [(n, "object") for n in pending]
    return out

def conj(f):
    if f[0] == "and": return [list(x) for x in f[1:]]
    return [list(f)]

def parse_domain(path):
    sx = read_sexp(tokenize(Path(path).read_text()))
    dom = {"name": sx[1][1], "requirements": [], "types": [], "predicates": {}, "actions": []}
    for sec in sx[2:]:
        key = sec[0]
        if key == ":requirements": dom["requirements"] = sec[1:]
        elif key == ":types": dom["types"] = [n for n, _ in typed_list(sec[1:])]
        elif key == ":predicates":
            for p in sec[1:]: dom["predicates"][p[0]] = typed_list(p[1:])
        elif key == ":action":
            a = {"name": sec[1]}
            for i in range(2, len(sec), 2):
                a[sec[i][1:]] = sec[i + 1]
            a["parameters"] = typed_list(a["parameters"])
            dom["actions"].append(a)
    return dom

def parse_problem(path):
    sx = read_sexp(tokenize(Path(path).read_text()))
    prob = {"name": sx[1][1]}
    for sec in sx[2:]:
        if sec[0] == ":objects": prob["objects"] = typed_list(sec[1:])
        elif sec[0] == ":init": prob["init"] = [tuple(f) for f in sec[1:]]
        elif sec[0] == ":goal": prob["goal"] = [tuple(f) for f in conj(sec[1])]
    return prob

dom = parse_domain("Domain.pddl")
assert set(dom["requirements"]) <= {":strips", ":typing"}, "Only :strips and :typing are allowed"
for a in dom["actions"]:
    assert all(x[0] != "not" for x in conj(a["precondition"])), "negative precondition found"
print("Domain:", dom["name"], "| requirements:", dom["requirements"])
print("Types:", dom["types"])
print("Predicates:", list(dom["predicates"]))
print("Actions:", [a["name"] for a in dom["actions"]])

Domain: mars-rover | requirements: [':strips', ':typing']
Types: ['location', 'rover', 'bay', 'rock', 'drill']
Predicates: ['rover-at', 'rock-at', 'drill-at', 'connected', 'soft', 'hard', 'has-drill', 'bay-empty', 'bay-holds']
Actions: ['move', 'pick-drill', 'collect-soft', 'collect-hard', 'deposit']


In [3]:
def ground(dom, prob):
    objs = prob["objects"]
    by_type = {}
    for n, t in objs: by_type.setdefault(t, []).append(n)
    init = set(prob["init"])
    dynamic = set()
    for a in dom["actions"]:
        for e in conj(a["effect"]):
            dynamic.add(e[1][0] if e[0] == "not" else e[0])
    facts = {}
    def fid(f):
        if f not in facts: facts[f] = len(facts)
        return facts[f]
    actions = []
    for a in dom["actions"]:
        names = [p for p, _ in a["parameters"]]
        pools = [by_type[t] for _, t in a["parameters"]]
        pre_l, eff_l = conj(a["precondition"]), conj(a["effect"])
        for combo in itertools.product(*pools):
            if len(set(combo)) != len(combo): continue
            sub = dict(zip(names, combo))
            g = lambda f: tuple([f[0]] + [sub.get(x, x) for x in f[1:]])
            ok, pre = True, []
            for p in pre_l:
                gp = g(p)
                if gp[0] in dynamic: pre.append(gp)
                elif gp not in init: ok = False; break      # static precondition false -> prune
            if not ok: continue
            add, dele = [], []
            for e in eff_l:
                if e[0] == "not": dele.append(g(e[1]))
                else: add.append(g(e))
            actions.append(("(" + " ".join([a["name"]] + list(combo)) + ")", pre, add, dele))
    for f in init:
        if f[0] in dynamic: fid(f)
    for nm, pre, add, dele in actions:
        for f in pre + add + dele: fid(f)
    for f in prob["goal"]: fid(f)
    gact = []
    for nm, pre, add, dele in actions:
        gact.append((nm, sum(1 << facts[f] for f in pre), sum(1 << facts[f] for f in add),
                     sum(1 << facts[f] for f in dele), [facts[f] for f in pre], [facts[f] for f in add]))
    s0 = sum(1 << facts[f] for f in init if f[0] in dynamic)
    gids = [facts[f] for f in prob["goal"]]
    return {"facts": facts, "actions": gact, "s0": s0,
            "gmask": sum(1 << i for i in gids), "gids": gids, "nf": len(facts)}

P1 = ground(dom, parse_problem("Problem.pddl"))
P2 = ground(dom, parse_problem("Problem_extended.pddl"))
print("Scenario 1: grounded actions =", len(P1["actions"]), "| dynamic facts =", P1["nf"])
print("Scenario 2: grounded actions =", len(P2["actions"]), "| dynamic facts =", P2["nf"])

Scenario 1: grounded actions = 162 | dynamic facts = 62
Scenario 2: grounded actions = 216 | dynamic facts = 80


In [4]:
INF = 10**9

def h_add(P, s):
    d = [INF] * P["nf"]
    for i in range(P["nf"]):
        if (s >> i) & 1: d[i] = 0
    changed = True
    while changed:
        changed = False
        for nm, pm, am, dm, pre, add in P["actions"]:
            c = 0
            for p in pre:
                c += d[p]
                if c >= INF: break
            if c >= INF: continue
            c += 1
            for a in add:
                if c < d[a]: d[a] = c; changed = True
    return sum(d[g] for g in P["gids"])

_LM = {}
def lm_cut(P, s):
    """LM-cut heuristic (admissible), unit action costs."""
    key = id(P)
    if key not in _LM:
        n = P["nf"]; G = n
        acts = [(pre, add) for (nm, pm, am, dm, pre, add) in P["actions"]] + [(P["gids"], [G])]
        pre_of = [[] for _ in range(n + 1)]; adders = [[] for _ in range(n + 1)]
        for i, (pre, add) in enumerate(acts):
            for f in pre: pre_of[f].append(i)
            for f in add: adders[f].append(i)
        _LM[key] = (n, G, acts, pre_of, adders)
    n, G, acts, pre_of, adders = _LM[key]
    na = len(acts)
    init = [i for i in range(n) if (s >> i) & 1]
    cost = [1] * na; cost[na - 1] = 0
    h = 0
    while True:
        d = [INF] * (n + 1); pcf = [-1] * na; cnt = [len(a[0]) for a in acts]
        pq = []
        for f in init: d[f] = 0; pq.append((0, f))
        heapq.heapify(pq); done = [False] * (n + 1)
        while pq:
            df, f = heapq.heappop(pq)
            if done[f]: continue
            done[f] = True
            for a in pre_of[f]:
                cnt[a] -= 1
                if cnt[a] == 0:
                    pcf[a] = f; ac = df + cost[a]
                    for e in acts[a][1]:
                        if ac < d[e]: d[e] = ac; heapq.heappush(pq, (ac, e))
        if d[G] >= INF: return INF
        if d[G] == 0: return h
        zone = {G}; st = [G]
        while st:
            f = st.pop()
            for a in adders[f]:
                if cost[a] == 0 and pcf[a] >= 0 and pcf[a] not in zone:
                    zone.add(pcf[a]); st.append(pcf[a])
        pidx = {}
        for a in range(na):
            if pcf[a] >= 0: pidx.setdefault(pcf[a], []).append(a)
        reach = set(init) - zone; st = list(reach); cut = set()
        while st:
            f = st.pop()
            for a in pidx.get(f, ()):
                for e in acts[a][1]:
                    if e in zone: cut.add(a)
                    elif e not in reach: reach.add(e); st.append(e)
        m = min(cost[a] for a in cut)
        h += m
        for a in cut: cost[a] -= m

def succ(P, s):
    for nm, pm, am, dm, pre, add in P["actions"]:
        if s & pm == pm: yield nm, (s & ~dm) | am

def build_plan(parent, s):
    plan = []
    while parent[s] is not None:
        ps, a = parent[s]; plan.append(a); s = ps
    return plan[::-1]

def bfs(P):
    t = time.time(); s0 = P["s0"]; g = P["gmask"]
    parent = {s0: None}; q = deque([s0]); exp = 0
    while q:
        s = q.popleft(); exp += 1
        if s & g == g: return build_plan(parent, s), exp, len(parent), time.time() - t
        for a, n in succ(P, s):
            if n not in parent: parent[n] = (s, a); q.append(n)
    return None, exp, len(parent), time.time() - t

def gbfs(P, h=h_add):
    t = time.time(); s0 = P["s0"]; g = P["gmask"]
    parent = {s0: None}; cnt = itertools.count(); pq = [(h(P, s0), next(cnt), s0)]
    exp = 0; closed = set()
    while pq:
        _, _, s = heapq.heappop(pq)
        if s in closed: continue
        closed.add(s); exp += 1
        if s & g == g: return build_plan(parent, s), exp, len(parent), time.time() - t
        for a, n in succ(P, s):
            if n not in parent:
                parent[n] = (s, a)
                hv = h(P, n)
                if hv < INF: heapq.heappush(pq, (hv, next(cnt), n))
    return None, exp, len(parent), time.time() - t

def astar(P, h=None, time_limit=None):
    h = h or lm_cut
    t = time.time(); s0 = P["s0"]; g = P["gmask"]
    gbest = {s0: 0}; parent = {s0: None}; cnt = itertools.count()
    pq = [(h(P, s0), 0, next(cnt), s0)]; closed = set(); exp = 0
    while pq:
        f, ngv, _, s = heapq.heappop(pq); gv = -ngv
        if s in closed: continue
        closed.add(s); exp += 1
        if s & g == g: return build_plan(parent, s), exp, len(parent), time.time() - t
        if exp % 2000 == 0:
            print(f"   ... A* progress: {exp} expanded, current f = {f}, {time.time()-t:.0f}s", flush=True)
            if time_limit and time.time() - t > time_limit:
                return None, exp, len(parent), time.time() - t
        for a, n in succ(P, s):
            ng = gv + 1
            if n not in gbest or ng < gbest[n]:
                gbest[n] = ng; parent[n] = (s, a)
                hv = h(P, n)
                if hv < INF: heapq.heappush(pq, (ng + hv, -ng, next(cnt), n))
    return None, exp, len(parent), time.time() - t

print("Search algorithms ready: bfs, gbfs (h_add), astar (LM-cut)")

Search algorithms ready: bfs, gbfs (h_add), astar (LM-cut)


## Task 3: Plan generation, configurations used

| Configuration | Algorithm | Heuristic | Guarantee |
|---|---|---|---|
| Satisficing | Greedy Best-First Search | h_add (not admissible) | valid plan, not necessarily shortest |
| Optimal | A* | LM-cut (admissible) | shortest plan |
| Optimal (cross-check, scenario 1) | Breadth-First Search | none | shortest plan |

All actions have unit cost, so "optimal" means the minimum number of actions.

In [5]:
def run(P, label, fn, **kw):
    plan, exp, gen, secs = fn(P, **kw)
    return {"label": label, "plan": plan, "expanded": exp, "generated": gen, "time": secs}

def show(res, P):
    print("=" * 78)
    print(res["label"])
    print("=" * 78)
    if res["plan"] is None:
        print("NOT FINISHED / no plan found.  States expanded so far:", res["expanded"],
              "| time: %.2fs" % res["time"])
        return
    for i, a in enumerate(res["plan"], 1): print(f"{i:2d}. {a}")
    print("Plan length (number of actions in plan):", len(res["plan"]))
    print("Total number of (grounded) actions in the problem:", len(P["actions"]))
    print("States expanded:", res["expanded"], "| states generated:", res["generated"])
    print("Planning time: %.2f s" % res["time"])

R1 = {}
R1["GBFS (satisficing)"] = run(P1, "SCENARIO 1 | Satisficing: Greedy Best-First Search + h_add", gbfs)
show(R1["GBFS (satisficing)"], P1)
R1["A* (optimal)"] = run(P1, "SCENARIO 1 | Optimal: A* + LM-cut", astar)
show(R1["A* (optimal)"], P1)
R1["BFS (optimal)"] = run(P1, "SCENARIO 1 | Optimal: Breadth-First Search", bfs)
show(R1["BFS (optimal)"], P1)

SCENARIO 1 | Satisficing: Greedy Best-First Search + h_add
 1. (move curiosity base sitea)
 2. (collect-soft curiosity rock1 sitea bay1)
 3. (move curiosity sitea base)
 4. (deposit curiosity rock1 base bay1)
 5. (move curiosity base siteb)
 6. (collect-soft curiosity rock2 siteb bay1)
 7. (move curiosity siteb base)
 8. (deposit curiosity rock2 base bay1)
 9. (move curiosity base sitea)
10. (move curiosity sitea sitec)
11. (move curiosity sitec sited)
12. (collect-soft curiosity rock4 sited bay1)
13. (move curiosity sited siteb)
14. (move curiosity siteb base)
15. (deposit curiosity rock4 base bay1)
16. (move curiosity base sitea)
17. (move curiosity sitea sitec)
18. (move curiosity sitec sitee)
19. (pick-drill curiosity drill1 sitee)
20. (move curiosity sitee sitec)
21. (collect-hard curiosity rock3 sitec bay1 drill1)
22. (move curiosity sitec sitea)
23. (move curiosity sitea base)
24. (deposit curiosity rock3 base bay1)
25. (move curiosity base sitea)
26. (move curiosity sitea sitec

In [6]:
def validate(P, plan, label):
    id2fact = {i: f for f, i in P["facts"].items()}
    amap = {a[0]: a for a in P["actions"]}
    def facts_of(state): return [id2fact[i] for i in range(P["nf"]) if (state >> i) & 1]
    def summary(state):
        fs = facts_of(state)
        rover = [f[2] for f in fs if f[0] == "rover-at"][0]
        bays = {}
        for f in fs:
            if f[0] == "bay-empty": bays[f[1]] = "empty"
        for f in fs:
            if f[0] == "bay-holds": bays[f[1]] = f[2]
        drill = "carried" if any(f[0] == "has-drill" for f in fs) else \
                "at " + [f[2] for f in fs if f[0] == "drill-at"][0]
        return f"rover@{rover} | " + " ".join(f"{b}={bays[b]}" for b in sorted(bays)) + f" | drill {drill}"
    print("=" * 78); print("VALIDATION:", label); print("=" * 78)
    state = P["s0"]
    print(" 0. (initial)".ljust(48), summary(state))
    for i, name in enumerate(plan, 1):
        nm, pm, am, dm, pre, add = amap[name]
        if state & pm != pm:
            print(f"INVALID: step {i} {name} is not applicable"); return False
        state = (state & ~dm) | am
        print(f"{i:2d}. {name}".ljust(48), summary(state))
    fs = facts_of(state)
    print("\nFinal state (dynamic facts):")
    for f in sorted(fs): print("   ", "(" + " ".join(f) + ")")
    goal_ok = (state & P["gmask"]) == P["gmask"]
    rocks_goal = [f for f in id2fact.values() if f[0] == "rock-at" and f[2] == "base"]
    rocks_at_base = sorted(f[1] for f in fs if f[0] == "rock-at" and f[2] == "base")
    bays_empty = sorted(f[1] for f in fs if f[0] == "bay-empty")
    rover_loc = [f[2] for f in fs if f[0] == "rover-at"][0]
    print("\nRocks at Base :", rocks_at_base)
    print("Empty bays    :", bays_empty)
    print("Rover at      :", rover_loc)
    print("GOAL SATISFIED:", goal_ok)
    return goal_ok

for k, r in R1.items():
    if r["plan"]: validate(P1, r["plan"], "Scenario 1 | " + k)

VALIDATION: Scenario 1 | GBFS (satisficing)
 0. (initial)                                    rover@base | bay1=empty bay2=empty | drill at sitee
 1. (move curiosity base sitea)                  rover@sitea | bay1=empty bay2=empty | drill at sitee
 2. (collect-soft curiosity rock1 sitea bay1)    rover@sitea | bay1=rock1 bay2=empty | drill at sitee
 3. (move curiosity sitea base)                  rover@base | bay1=rock1 bay2=empty | drill at sitee
 4. (deposit curiosity rock1 base bay1)          rover@base | bay1=empty bay2=empty | drill at sitee
 5. (move curiosity base siteb)                  rover@siteb | bay1=empty bay2=empty | drill at sitee
 6. (collect-soft curiosity rock2 siteb bay1)    rover@siteb | bay1=rock2 bay2=empty | drill at sitee
 7. (move curiosity siteb base)                  rover@base | bay1=rock2 bay2=empty | drill at sitee
 8. (deposit curiosity rock2 base bay1)          rover@base | bay1=empty bay2=empty | drill at sitee
 9. (move curiosity base sitea)            

## Task 4: Goal verification

Each plan was simulated step by step: every action's preconditions were checked in the current state, then its delete and add effects were applied.
The printed final state shows all rocks at Base, both bays empty and the rover at Base, and `GOAL SATISFIED: True` confirms the goal.
(The simulator reuses the grounded actions. As an independent check you can also run the VAL validator on `Domain.pddl`, `Problem.pddl` and a plan file, if you have VAL installed.)

In [7]:
TIME_LIMIT = 1800   # seconds allowed for the optimal A* search in the extended scenario

R2 = {}
R2["GBFS (satisficing)"] = run(P2, "SCENARIO 2 (extended) | Satisficing: Greedy Best-First Search + h_add", gbfs)
show(R2["GBFS (satisficing)"], P2)
validate(P2, R2["GBFS (satisficing)"]["plan"], "Scenario 2 | GBFS (satisficing)")

print("\nRunning optimal A* + LM-cut on the extended scenario (this can take several minutes)...")
R2["A* (optimal)"] = run(P2, "SCENARIO 2 (extended) | Optimal: A* + LM-cut", astar, time_limit=TIME_LIMIT)
show(R2["A* (optimal)"], P2)
if R2["A* (optimal)"]["plan"]:
    validate(P2, R2["A* (optimal)"]["plan"], "Scenario 2 | A* (optimal)")

SCENARIO 2 (extended) | Satisficing: Greedy Best-First Search + h_add
 1. (move curiosity base sitea)
 2. (collect-soft curiosity rock1 sitea bay1)
 3. (move curiosity sitea base)
 4. (deposit curiosity rock1 base bay1)
 5. (move curiosity base sitea)
 6. (move curiosity sitea sitec)
 7. (move curiosity sitec sitee)
 8. (pick-drill curiosity drill1 sitee)
 9. (move curiosity sitee sitec)
10. (collect-hard curiosity rock3 sitec bay1 drill1)
11. (move curiosity sitec sitea)
12. (move curiosity sitea base)
13. (deposit curiosity rock3 base bay1)
14. (move curiosity base sitea)
15. (move curiosity sitea sitec)
16. (move curiosity sitec sitee)
17. (collect-soft curiosity rock6 sitee bay1)
18. (move curiosity sitee sitec)
19. (move curiosity sitec sitea)
20. (move curiosity sitea base)
21. (deposit curiosity rock6 base bay1)
22. (move curiosity base sitea)
23. (move curiosity sitea sitec)
24. (move curiosity sitec sited)
25. (collect-soft curiosity rock4 sited bay1)
26. (move curiosity sited

In [8]:
def wastar(P, w=2.0):
    t = time.time(); s0 = P["s0"]; g = P["gmask"]
    gb = {s0: 0}; parent = {s0: None}; cnt = itertools.count()
    pq = [(w * lm_cut(P, s0), 0, next(cnt), s0)]; closed = set(); exp = 0
    while pq:
        f, ngv, _, s = heapq.heappop(pq); gv = -ngv
        if s in closed: continue
        closed.add(s); exp += 1
        if s & g == g: return build_plan(parent, s), exp, len(parent), time.time() - t
        for a, n in succ(P, s):
            ng = gv + 1
            if n not in gb or ng < gb[n]:
                gb[n] = ng; parent[n] = (s, a); hv = lm_cut(P, n)
                if hv < INF: heapq.heappush(pq, (ng + w * hv, -ng, next(cnt), n))
    return None, exp, len(parent), time.time() - t

OPT2 = "A* w=2 (verified optimal)"
R2[OPT2] = run(P2, "SCENARIO 2 (extended) | Weighted A* (w=2, LM-cut); length proven optimal by exhaustive BFS below", wastar, w=2.0)
show(R2[OPT2], P2)
validate(P2, R2[OPT2]["plan"], "Scenario 2 | " + OPT2)

SCENARIO 2 (extended) | Weighted A* (w=2, LM-cut); length proven optimal by exhaustive BFS below
 1. (move curiosity base sitea)
 2. (collect-soft curiosity rock1 sitea bay1)
 3. (move curiosity sitea base)
 4. (deposit curiosity rock1 base bay1)
 5. (move curiosity base sitea)
 6. (move curiosity sitea sitec)
 7. (move curiosity sitec sited)
 8. (collect-soft curiosity rock4 sited bay1)
 9. (move curiosity sited siteb)
10. (collect-soft curiosity rock2 siteb bay2)
11. (move curiosity siteb sited)
12. (move curiosity sited sitee)
13. (pick-drill curiosity drill1 sitee)
14. (move curiosity sitee sitec)
15. (move curiosity sitec sitea)
16. (move curiosity sitea base)
17. (deposit curiosity rock2 base bay2)
18. (deposit curiosity rock4 base bay1)
19. (move curiosity base sitea)
20. (move curiosity sitea sitec)
21. (move curiosity sitec sitee)
22. (move curiosity sitee sitef)
23. (collect-hard curiosity rock5 sitef bay1 drill1)
24. (collect-hard curiosity rock7 sitef bay2 drill1)
25. (move

True

In [9]:
%%writefile bfs_proof.cpp
#include <bits/stdc++.h>
using namespace std;
// Exhaustive BFS over the extended-scenario state space.
// Locations: 0 Base,1 A,2 B,3 C,4 D,5 E,6 F.  Rock position: 0-6 = location, 7 = Bay1, 8 = Bay2.
int NR=7;
vector<pair<int,int>> edges={{0,1},{1,0},{1,3},{3,1},{2,4},{4,2},{3,5},{5,3},{4,5},{5,4},{5,6},{6,5},{3,4}}; // Base<->B closed, C->D one-way
bool hard[7]={0,0,1,0,1,0,1};  // rock3, rock5, rock7 are hard
typedef unsigned long long u64;
u64 enc(int rv,int dr,int*r){u64 x=rv*2+dr;for(int i=0;i<NR;i++)x=x*9+r[i];return x;}
void dec(u64 x,int&rv,int&dr,int*r){for(int i=NR-1;i>=0;i--){r[i]=x%9;x/=9;}dr=x%2;x/=2;rv=x;}
int main(){
  int init[7]={1,2,3,4,6,5,6}; // rock1 A, rock2 B, rock3 C, rock4 D, rock5 F, rock6 E, rock7 F
  u64 total=7*2; for(int i=0;i<NR;i++) total*=9;
  vector<signed char> dist(total,-1);
  u64 s0=enc(0,0,init); dist[s0]=0;
  vector<u64> cur={s0},nxt; int d=0; u64 count=1;
  while(!cur.empty()){
    for(u64 s:cur){
      int rv,dr,r[7]; dec(s,rv,dr,r);
      bool goal=(rv==0); for(int i=0;i<NR;i++) if(r[i]!=0) goal=false;
      if(goal){printf("GOAL first reached at depth %d (shortest possible plan length). States visited: %llu\n",d,count);return 0;}
      auto push=[&](int rv2,int dr2,int*r2){u64 t=enc(rv2,dr2,r2); if(dist[t]<0){dist[t]=d+1;nxt.push_back(t);count++;}};
      int b1=-1,b2=-1; for(int i=0;i<NR;i++){if(r[i]==7)b1=i; if(r[i]==8)b2=i;}
      for(auto&e:edges) if(e.first==rv) push(e.second,dr,r);
      if(dr==0&&rv==5) push(rv,1,r);
      for(int i=0;i<NR;i++){
        if(r[i]==rv && (!hard[i]||dr==1)){
          if(b1<0){int q[7];memcpy(q,r,sizeof q);q[i]=7;push(rv,dr,q);}
          if(b2<0){int q[7];memcpy(q,r,sizeof q);q[i]=8;push(rv,dr,q);}
        }
        if(r[i]==7||r[i]==8){int q[7];memcpy(q,r,sizeof q);q[i]=rv;push(rv,dr,q);}
      }
    }
    cur.swap(nxt); nxt.clear(); d++;
  }
  printf("no goal reachable\n");
}

Writing bfs_proof.cpp


In [10]:
!g++ -O2 -o bfs_proof bfs_proof.cpp && ./bfs_proof

GOAL first reached at depth 40 (shortest possible plan length). States visited: 22274690


In [11]:
OPT2 = "A* w=2 (verified optimal)"

def table(R, P, title):
    print(title)
    print(f"{'Configuration':<28}{'Plan length':>12}{'Grounded actions':>18}{'Expanded':>12}{'Generated':>12}{'Time (s)':>10}")
    for k, r in R.items():
        pl = len(r["plan"]) if r["plan"] else "n/a"
        print(f"{k:<28}{str(pl):>12}{len(P['actions']):>18}{r['expanded']:>12}{r['generated']:>12}{r['time']:>10.2f}")
    print()

table(R1, P1, "SCENARIO 1 (original)")
table(R2, P2, "SCENARIO 2 (extended, dust storm)")
print("Note: plain A*+LM-cut on Scenario 2 did not finish within the 1800 s limit (204,000 states expanded).")
print("Exhaustive BFS shows the optimal length is 40.\n")

for name, g, a in [("Scenario 1", R1["GBFS (satisficing)"]["plan"], R1["A* (optimal)"]["plan"]),
                   ("Scenario 2", R2["GBFS (satisficing)"]["plan"], R2[OPT2]["plan"])]:
    print(f"{name}: GBFS = {len(g)}, optimal = {len(a)}, GBFS is {len(g)-len(a)} actions longer ({(len(g)-len(a))/len(a)*100:.0f}%)")

SCENARIO 1 (original)
Configuration                Plan length  Grounded actions    Expanded   Generated  Time (s)
GBFS (satisficing)                    34               162          78         221      0.04
A* (optimal)                          25               162        6741       13781     34.88
BFS (optimal)                         25               162      332520      337038      6.96

SCENARIO 2 (extended, dust storm)
Configuration                Plan length  Grounded actions    Expanded   Generated  Time (s)
GBFS (satisficing)                    61               216         722        1652      0.37
A* (optimal)                         n/a               216      204000      418628   1805.95
A* w=2 (verified optimal)             40               216        2425        6438     23.32

Note: plain A*+LM-cut on Scenario 2 did not finish within the 1800 s limit (204,000 states expanded).
Exhaustive BFS shows the optimal length is 40.

Scenario 1: GBFS = 34, optimal = 25, GBFS is 9 a

In [12]:
def analyse(plan, label):
    print("=" * 78); print("ANALYSIS:", label); print("=" * 78)
    bays = {"bay1": None, "bay2": None}
    max_carried, drill_step = 0, None
    for i, a in enumerate(plan, 1):
        t = a.strip("()").split()
        if t[0] == "pick-drill" and drill_step is None:
            drill_step = i
            print(f"Step {i:2d}: DRILL picked up at {t[3]}  (next action must be a move; rover now has the drill permanently)")
        if t[0] in ("collect-soft", "collect-hard"):
            rock, loc, bay = t[2], t[3], t[4]
            bays[bay] = rock
            print(f"Step {i:2d}: {bay} <- {rock} (collected at {loc}, {t[0]})")
        if t[0] == "deposit":
            rock, loc, bay = t[2], t[3], t[4]
            bays[bay] = None
            print(f"Step {i:2d}: {bay} -> {rock} deposited at {loc}")
        max_carried = max(max_carried, sum(v is not None for v in bays.values()))
    used_both_together = max_carried == 2
    print(f"\nMaximum rocks carried at once: {max_carried}  (both bays used simultaneously: {used_both_together})")
    print("Drill picked up at step:", drill_step, "of", len(plan))
    before = [a for a in plan[:drill_step - 1] if a.startswith("(collect-hard")] if drill_step else []
    print("Hard rocks collected before the drill was picked:", before, "(must be empty)")
    return drill_step, max_carried

def moves(plan):
    return [tuple(a.strip("()").split()[2:4]) for a in plan if a.startswith("(move")]

print("######## TASK 6.2: storage bays and drill (original scenario) ########")
analyse(R1["A* (optimal)"]["plan"], "Scenario 1 | optimal plan (A*)")
analyse(R1["GBFS (satisficing)"]["plan"], "Scenario 1 | satisficing plan (GBFS), for comparison")

print("\n######## TASK 6.3: how SiteB and SiteD are reached after the dust storm ########")
for key in [OPT2, "GBFS (satisficing)"]:
    pl = R2[key]["plan"]
    if not pl: print(key, ": plan not available"); continue
    mv = moves(pl)
    print(f"\n--- {key} ---")
    print("Entries into SiteB:", [f"{a}->{b}" for a, b in mv if b == "siteb"])
    print("Entries into SiteD:", [f"{a}->{b}" for a, b in mv if b == "sited"])
    print("Uses one-way slope SiteC->SiteD:", ("sitec", "sited") in mv, "| times:", mv.count(("sitec", "sited")))
    print("Any move involving Base<->SiteB (should be none):", [m for m in mv if set(m) == {"base", "siteb"}])
    print("Moves SiteD->SiteC (impossible, should be none):", [m for m in mv if m == ("sited", "sitec")])

######## TASK 6.2: storage bays and drill (original scenario) ########
ANALYSIS: Scenario 1 | optimal plan (A*)
Step  2: bay1 <- rock2 (collected at siteb, collect-soft)
Step  4: bay2 <- rock4 (collected at sited, collect-soft)
Step  7: bay1 -> rock2 deposited at base
Step  8: bay2 -> rock4 deposited at base
Step 12: DRILL picked up at sitee  (next action must be a move; rover now has the drill permanently)
Step 14: bay1 <- rock5 (collected at sitef, collect-hard)
Step 17: bay2 <- rock3 (collected at sitec, collect-hard)
Step 20: bay2 -> rock3 deposited at base
Step 21: bay1 -> rock5 deposited at base
Step 23: bay1 <- rock1 (collected at sitea, collect-soft)
Step 25: bay1 -> rock1 deposited at base

Maximum rocks carried at once: 2  (both bays used simultaneously: True)
Drill picked up at step: 12 of 25
Hard rocks collected before the drill was picked: [] (must be empty)
ANALYSIS: Scenario 1 | satisficing plan (GBFS), for comparison
Step  2: bay1 <- rock1 (collected at sitea, collect-s

In [13]:
_ = analyse(R2[OPT2]["plan"], "Scenario 2 | optimal plan")

ANALYSIS: Scenario 2 | optimal plan
Step  2: bay1 <- rock1 (collected at sitea, collect-soft)
Step  4: bay1 -> rock1 deposited at base
Step  8: bay1 <- rock4 (collected at sited, collect-soft)
Step 10: bay2 <- rock2 (collected at siteb, collect-soft)
Step 13: DRILL picked up at sitee  (next action must be a move; rover now has the drill permanently)
Step 17: bay2 -> rock2 deposited at base
Step 18: bay1 -> rock4 deposited at base
Step 23: bay1 <- rock5 (collected at sitef, collect-hard)
Step 24: bay2 <- rock7 (collected at sitef, collect-hard)
Step 29: bay1 -> rock5 deposited at base
Step 30: bay2 -> rock7 deposited at base
Step 33: bay1 <- rock3 (collected at sitec, collect-hard)
Step 35: bay2 <- rock6 (collected at sitee, collect-soft)
Step 39: bay1 -> rock3 deposited at base
Step 40: bay2 -> rock6 deposited at base

Maximum rocks carried at once: 2  (both bays used simultaneously: True)
Drill picked up at step: 13 of 40
Hard rocks collected before the drill was picked: [] (must be e

In [14]:
import io, contextlib

buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    for R, P, nm in [(R1, P1, "SCENARIO 1"), (R2, P2, "SCENARIO 2 (extended)")]:
        for k, r in R.items():
            show(r, P)
        table(R, P, nm + " summary")
Path("planner_output.txt").write_text(buf.getvalue())

for R, fn in [(R1, "plan_scenario1_{}.txt"), (R2, "plan_scenario2_{}.txt")]:
    for k, r in R.items():
        if r["plan"]:
            tag = "gbfs" if k.startswith("GBFS") else ("astar" if k.startswith("A*") else "bfs")
            Path(fn.format(tag)).write_text("\n".join(r["plan"]) + f"\n; cost = {len(r['plan'])}\n")
print("Saved: planner_output.txt and plan_*.txt")
print([str(p) for p in Path('.').glob('*.pddl')] + [str(p) for p in Path('.').glob('plan_*.txt')] + ["planner_output.txt"])

Saved: planner_output.txt and plan_*.txt
['Domain.pddl', 'Problem.pddl', 'Problem_extended.pddl', 'plan_scenario1_astar.txt', 'plan_scenario2_gbfs.txt', 'plan_scenario2_astar.txt', 'plan_scenario1_gbfs.txt', 'plan_scenario1_bfs.txt', 'planner_output.txt']
